In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
import operator
from typing import TypedDict, Annotated

from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import StateGraph, START, END
import sqlite3

In [2]:
class MyState(TypedDict):
    messages: Annotated[list, operator.add]


def node_1(state: MyState):
    return {"messages": ["abc", "def"]}

In [3]:
# 数据存储到sqlite_data目录下面，需要目录存在
conn = sqlite3.connect(database="./sqlite_data/langgraph_sqlite", check_same_thread=False)
memory = SqliteSaver(conn=conn)

builder = StateGraph(MyState)
builder.add_node("node_1", node_1)
builder.add_edge(START, "node_1")
builder.add_edge("node_1", END)

graph = builder.compile(checkpointer=memory)

config = {"configurable": {"thread_id": "1"}}

initial_state = graph.get_state(config)
print(f"Initial state: {initial_state}")

Initial state: StateSnapshot(values={'messages': ['abc', 'def']}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0e60ea-775e-6b2d-8001-ffe764c62c6f'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2025-12-31T06:05:06.193898+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f0e60ea-775c-6514-8000-d06573ad6626'}}, tasks=(), interrupts=())


In [4]:
# 执行图
result = graph.invoke({"messages": []}, config)
print(f"Result: {result}")

Result: {'messages': ['abc', 'def', 'abc', 'def']}


In [5]:
# 查看执行后的状态
final_state = graph.get_state(config)
print(f"Final state: {final_state}")

conn.close()

Final state: StateSnapshot(values={'messages': ['abc', 'def', 'abc', 'def']}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f15b62f-1f18-6cf9-8004-bc0fb1f4fc73'}}, metadata={'source': 'loop', 'step': 4, 'parents': {}}, created_at='2026-05-29T13:33:14.925797+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f15b62f-1f13-6ee1-8003-aa9c8a57d821'}}, tasks=(), interrupts=())
